# Gateway / Agent / Runner 源码走读

本节深入 OpenClaw 三大核心组件的实现细节，理解请求从进入到执行的完整链路。

## 三大组件职责

### 1. Gateway（网关）
- **API 入口**: 统一接收来自 Web、APP、开放平台的请求
- **路由分发**: 根据请求类型分配到不同的 Agent 实例
- **鉴权限流**: Token 验证、QPS 限流、权限校验
- **协议转换**: HTTP/WebSocket/gRPC 统一转为内部消息格式

### 2. Agent（智能体）
- **状态管理**: 维护对话上下文、会话历史
- **意图理解**: 调用 LLM 解析用户意图
- **决策编排**: 决定调用哪些插件、以什么顺序执行
- **记忆系统**: 短期（会话内）+ 长期（跨会话）记忆

### 3. Runner（运行器）
- **执行引擎**: 按照 Agent 的决策计划执行具体操作
- **工具调用**: 管理插件的加载、调用、超时、重试
- **结果聚合**: 收集多个插件的返回结果并合并
- **Loop 控制**: 支持 ReAct、Plan-Execute 等循环模式

在B站商业化场景中，一个典型的广告效果查询请求会依次经过这三层处理。

In [ ]:
# 模拟 Gateway -> Agent -> Runner 的请求处理管线

from dataclasses import dataclass, field
from typing import Any
import json


@dataclass
class Request:
    """统一请求格式（Gateway 协议转换后的内部格式）"""
    user_id: str
    query: str
    channel: str  # web / app / feishu
    metadata: dict = field(default_factory=dict)


class Gateway:
    """网关：鉴权 + 限流 + 路由"""

    RATE_LIMIT = 100  # 每秒最大请求数
    VALID_TOKENS = {"token_biz_001", "token_biz_002"}

    def authenticate(self, token: str) -> bool:
        """Token 鉴权"""
        return token in self.VALID_TOKENS

    def route(self, request: Request) -> str:
        """根据 query 内容路由到对应 Agent"""
        if "广告" in request.query:
            return "ad_agent"
        elif "推荐" in request.query:
            return "rec_agent"
        return "general_agent"

    def handle(self, raw_request: dict) -> dict:
        """网关入口"""
        # 1. 鉴权
        if not self.authenticate(raw_request.get("token", "")):
            return {"error": "认证失败", "code": 401}

        # 2. 协议转换
        req = Request(
            user_id=raw_request["user_id"],
            query=raw_request["query"],
            channel=raw_request.get("channel", "web"),
        )

        # 3. 路由
        agent_name = self.route(req)
        print(f"[Gateway] 鉴权通过 -> 路由到 {agent_name}")
        return {"agent": agent_name, "request": req}


class Agent:
    """智能体：状态管理 + 意图解析 + 决策"""

    def __init__(self, name: str):
        self.name = name
        self.context = []  # 对话上下文

    def parse_intent(self, query: str) -> dict:
        """模拟 LLM 意图解析（实际会调用大模型）"""
        if "效果" in query or "数据" in query:
            return {"intent": "query_performance", "plugins": ["ad_query"]}
        elif "创建" in query or "投放" in query:
            return {"intent": "create_campaign", "plugins": ["ad_create", "budget_check"]}
        return {"intent": "general_qa", "plugins": ["knowledge_base"]}

    def decide(self, request: Request) -> dict:
        """决策：生成执行计划"""
        self.context.append({"role": "user", "content": request.query})
        intent = self.parse_intent(request.query)
        plan = {
            "intent": intent["intent"],
            "plugins": intent["plugins"],
            "params": {"query": request.query, "user_id": request.user_id},
        }
        print(f"[Agent:{self.name}] 意图={intent['intent']}, 计划调用插件={intent['plugins']}")
        return plan


# 执行演示
gateway = Gateway()
agent = Agent("ad_agent")

raw = {"user_id": "advertiser_88", "query": "查询我的广告投放效果数据",
       "token": "token_biz_001", "channel": "web"}

result = gateway.handle(raw)
if "error" not in result:
    plan = agent.decide(result["request"])
    print(f"[决策结果] {json.dumps(plan, ensure_ascii=False, indent=2)}")

In [ ]:
# Runner：执行引擎 -- 管理工具调用的完整生命周期

from enum import Enum


class ToolStatus(Enum):
    PENDING = "pending"
    RUNNING = "running"
    SUCCESS = "success"
    FAILED = "failed"
    TIMEOUT = "timeout"


class Runner:
    """执行引擎：插件调用 + 生命周期管理 + 结果聚合"""

    TIMEOUT_SEC = 30  # 插件执行超时时间
    MAX_RETRIES = 2   # 最大重试次数

    def __init__(self):
        self.plugin_registry = {}  # 已注册的插件
        self.execution_log = []    # 执行日志

    def register_plugin(self, name: str, plugin_fn):
        """注册插件"""
        self.plugin_registry[name] = plugin_fn

    def execute_plugin(self, name: str, params: dict) -> dict:
        """执行单个插件（含重试逻辑）"""
        if name not in self.plugin_registry:
            return {"status": ToolStatus.FAILED, "error": f"插件 {name} 未注册"}

        for attempt in range(self.MAX_RETRIES + 1):
            try:
                status = ToolStatus.RUNNING
                self.execution_log.append(f"  [{name}] 第{attempt+1}次执行 - {status.value}")
                result = self.plugin_registry[name](params)
                status = ToolStatus.SUCCESS
                self.execution_log.append(f"  [{name}] 执行完成 - {status.value}")
                return {"status": status, "data": result}
            except Exception as e:
                status = ToolStatus.FAILED
                self.execution_log.append(f"  [{name}] 第{attempt+1}次失败: {e}")
                if attempt == self.MAX_RETRIES:
                    return {"status": status, "error": str(e)}

    def run(self, plan: dict) -> dict:
        """按计划执行所有插件并聚合结果"""
        self.execution_log.append(f"Runner 开始执行计划: intent={plan['intent']}")
        results = {}
        for plugin_name in plan["plugins"]:
            result = self.execute_plugin(plugin_name, plan["params"])
            results[plugin_name] = result
        self.execution_log.append("Runner 执行完成, 聚合结果")
        return results


# 注册模拟插件
runner = Runner()

def mock_ad_query(params):
    return {"campaign": "双11大促", "impressions": 150000, "clicks": 4800, "ctr": "3.2%", "cost": "¥23,700"}

runner.register_plugin("ad_query", mock_ad_query)

# 使用上一步 Agent 生成的 plan
plan = {"intent": "query_performance", "plugins": ["ad_query"],
        "params": {"query": "查询广告效果", "user_id": "advertiser_88"}}

results = runner.run(plan)

print("=== Runner 执行日志 ===")
for log in runner.execution_log:
    print(log)

print("\n=== 执行结果 ===")
for name, res in results.items():
    print(f"  {name}: status={res['status'].value}, data={res.get('data', res.get('error'))}")

## 源码走读要点

### Gateway 关键实现
```
Gateway
 ├── middleware_chain    # 中间件链（鉴权、日志、限流）
 ├── router             # 路由表（path -> agent 映射）
 └── protocol_adapter   # 协议适配（HTTP/WS/gRPC -> 内部格式）
```

### Agent 关键实现
```
Agent
 ├── memory_store       # 记忆存储（短期 context + 长期 vector DB）
 ├── llm_client         # 大模型调用客户端
 ├── planner            # 规划器（生成执行计划）
 └── state_machine      # 状态机（管理对话状态转移）
```

### Runner 关键实现
```
Runner
 ├── plugin_registry    # 插件注册表
 ├── executor           # 执行器（同步/异步/并行）
 ├── retry_policy       # 重试策略
 ├── timeout_manager    # 超时管理
 └── result_aggregator  # 结果聚合器
```

## 面试速记

### Q1: Gateway 如何实现多协议支持？
**答**: 通过 Protocol Adapter 模式，将 HTTP/WebSocket/gRPC 请求统一转为内部 `Request` 对象。
Gateway 本身不关心协议细节，只处理标准化后的请求。

### Q2: Agent 的状态管理是怎么做的？
**答**: 短期记忆（当前会话上下文）存在内存中，长期记忆（用户偏好、历史行为）存在向量数据库中。
每次决策时，Agent 会从两种记忆中检索相关信息，拼接到 LLM 的 prompt 中。

### Q3: Runner 如何处理插件超时和失败？
**答**: Runner 实现了完整的生命周期管理：
- **超时控制**: 每个插件调用有独立的 timeout 设置
- **重试策略**: 支持指数退避重试，可配置最大重试次数
- **降级方案**: 核心插件失败时可 fallback 到缓存数据或默认响应

### Q4: 这三个组件之间的通信方式？
**答**: 同进程内通过方法调用（低延迟），跨进程时通过消息队列（如 Kafka）。
B站在高并发场景下使用异步消息解耦 Gateway 和 Agent。

> **加分项**: 能结合B站实际场景（如广告竞价时 Runner 需要毫秒级响应）说明设计权衡。